In [2]:
import pandas as pd
com = pd.read_csv('OrderFiles/com_orders.csv')
ie = pd.read_csv('OrderFiles/irish_orders.csv')
uk = pd.read_csv('OrderFiles/uk_orders.csv')

com['Name'] = com['Name'].str.lstrip('#').astype(int) + 800000
ie['Name'] = ie['Name'].str.lstrip('#').astype(int) + 100000
uk['Name'] = uk['Name'].str.lstrip('#').astype(int) + 300000

com['Name'] = com['Name'].apply(lambda x: f"#{x}")
ie['Name'] = ie['Name'].apply(lambda x: f"#{x}")
uk['Name'] = uk['Name'].apply(lambda x: f"#{x}")

df = pd.concat([com, ie, uk])
df.to_csv('all_orders.csv', index=False)

In [3]:
df = pd.read_csv('all_orders.csv', header=None)

C:\Users\darag\AppData\Local\Temp\ipykernel_7588\1752361789.py:1: DtypeWarning: Columns (8,9,10,11,12,13,16,18,19,21,22,44,45,46,49,51,52,55,57,58,59,60,61,62,63,64,65,66,67,68,72,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('all_orders.csv', header=None)


In [4]:
INPUT_COLUMNS = {
    0: 'order_id',
    1: 'email',
    3: 'order_date',
    17: 'item_name',
    24: 'customer_name',
    36: 'address_line1',
    42: 'country_code',
    11: 'total_spent'
}

replacements_df = pd.read_csv('item_name_replacements.csv')
ITEM_NAME_REPLACEMENTS = dict(zip(replacements_df['original_name'], replacements_df['new_name']))

df = df[list(INPUT_COLUMNS.keys())].rename(columns=INPUT_COLUMNS)

In [5]:
def normalize_name(name):
    if not isinstance(name, str) or pd.isna(name):
        return ''
    parts = [p for p in name.split() if p.lower() not in ['mr', 'mrs', 'ms']]
    return ' '.join(parts).title()

df['customer_name'] = df['customer_name'].apply(normalize_name)
df['email'] = df['email'].str.lower().str.strip()
df['address_line1'] = df['address_line1'].fillna('').str.title().str.strip()
df['total_spent'] = pd.to_numeric(df['total_spent'], errors='coerce').fillna(0)
df['order_date'] = pd.to_datetime(df['order_date'], utc=True, errors='coerce').dt.strftime('%Y-%m-%d')
df['country_code'] = df['country_code'].fillna('').astype(str).str.upper()
df['item_name'] = df['item_name'].replace(ITEM_NAME_REPLACEMENTS)

df_expanded = df.assign(email=df['email'].str.split('; ')).explode('email')
df_expanded['email'] = df_expanded['email'].str.strip().fillna('')

order_details = (df_expanded.groupby('order_id')
    .agg({
        'order_date': 'first',
        'item_name': lambda x: ', '.join(str(val) for val in x),
        'total_spent': 'first',
        'customer_name': 'first',
        'email': 'first',
        'address_line1': 'first',
        'country_code': 'first'
    })
    .reset_index())

order_details['order_details'] = order_details.apply(
    lambda row: f"Date:{row['order_date']} - Product(s):{row['item_name']} - Spent:({row['total_spent']})",
    axis=1
)

C:\Users\darag\AppData\Local\Temp\ipykernel_7588\431525884.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['order_date'] = pd.to_datetime(df['order_date'], utc=True, errors='coerce').dt.strftime('%Y-%m-%d')


In [6]:
def find_connected_customers(df):
    identifier_to_orders = {}
    for idx, row in df.iterrows():
        name_country = f"{row['customer_name']}|{row['country_code']}" if row['customer_name'] else ''
        email_country = f"{row['email']}|{row['country_code']}" if row['email'] else ''
        address_country = f"{row['address_line1']}|{row['country_code']}" if row['address_line1'] else ''
        
        if name_country:
            identifier_to_orders.setdefault(name_country, []).append(idx)
        if email_country:
            identifier_to_orders.setdefault(email_country, []).append(idx)
        if address_country:
            identifier_to_orders.setdefault(address_country, []).append(idx)
    
    customer_groups = {}
    customer_id = 0
    
    for idx in df.index:
        if not any(idx in orders for orders in identifier_to_orders.values()):
            customer_id += 1
            customer_groups[idx] = customer_id
    
    for identifier, order_indices in identifier_to_orders.items():
        if not order_indices or identifier == '':
            continue
        group = set()
        for idx in order_indices:
            if idx not in customer_groups:
                customer_id += 1
                customer_groups[idx] = customer_id
            group.add(customer_groups[idx])
        target_id = min(group)
        for idx in order_indices:
            customer_groups[idx] = target_id
    
    df['customer_id'] = df.index.map(lambda x: customer_groups.get(x, customer_id + 1 + x))
    return df

df_with_ids = find_connected_customers(order_details)

In [7]:
orders_per_customer = (df_with_ids.groupby('customer_id')
    .agg({
        'order_id': 'count',
        'total_spent': 'sum',
        'customer_name': lambda x: pd.Series(x).value_counts().idxmax() if not x.dropna().empty else '',
        'email': lambda x: pd.Series(x).value_counts().idxmax() if not x.dropna().empty else '',
        'address_line1': lambda x: pd.Series(x).value_counts().idxmax() if not x.dropna().empty else '',
        'country_code': lambda x: pd.Series(x).value_counts().idxmax() if not x.dropna().empty else '',
        'order_details': list
    })
    .rename(columns={
        'order_id': 'number_of_orders',
        'customer_name': 'most_used_name',
        'email': 'most_used_email',
        'address_line1': 'most_used_address',
        'country_code': 'most_used_country',
        'order_details': 'orders_list'
    })
    .reset_index(drop=True))

orders_per_customer = orders_per_customer.sort_values(
    ['number_of_orders', 'most_used_name'],
    ascending=[False, True]
)

max_orders = orders_per_customer['number_of_orders'].max()
for i in range(max_orders):
    orders_per_customer[f'order_{i+1}'] = orders_per_customer['orders_list'].apply(
        lambda x: x[i] if i < len(x) else ''
    )

orders_per_customer = orders_per_customer.drop(columns=['orders_list'])


In [8]:
def split_name(name):
    parts = name.split(' ', 1)
    return parts[0], parts[1] if len(parts) > 1 else ''

orders_per_customer[['first_name', 'last_name']] = pd.DataFrame(
    orders_per_customer['most_used_name'].apply(split_name).tolist(), 
    index=orders_per_customer.index
)

base_columns = [
    'first_name',
    'last_name',
    'most_used_email',
    'most_used_address',
    'most_used_country',
    'number_of_orders',
    'total_spent'
]

order_columns = [f'order_{i+1}' for i in range(max_orders)]
orders_per_customer = orders_per_customer[base_columns + order_columns]
orders_per_customer['total_spent'] = orders_per_customer['total_spent'].round(2)
orders_per_customer = orders_per_customer[orders_per_customer['total_spent'] != 0.0]

orders_per_customer.to_csv('customer_database.csv', index=False)

In [11]:
repeat_customers = orders_per_customer[orders_per_customer['number_of_orders'] > 1]
repeat_buying_rate = len(repeat_customers) / len(orders_per_customer) * 100
print(f"Total customers: {len(orders_per_customer)}")
print(f"Repeat customers: {len(repeat_customers)}")
print(f"Repeat buying rate: {repeat_buying_rate:.2f}%")

Total customers: 7753
Repeat customers: 2166
Repeat buying rate: 27.94%
